# Phase 1 Final Dedicated Controls

This notebook is orchestration only. It runs the repository CLI for dedicated final negative controls after the final feature matrix and final comparator reconciliation have been reviewed.

Scientific integrity rules:

- This notebook does not train claim-bearing comparators and does not open headline claims.
- The runner computes only dedicated negative controls from reviewed final feature-matrix rows and locked LOSO folds.
- All fitting must use training subjects only; outer-test subjects must not enter normalization, nuisance, spatial, teacher, privileged, gate, or weight fits.
- Test-time inference must remain scalp-only. Teacher/proxy outputs are not allowed at test time.
- Failed controls are valid results and must remain blockers. Do not change thresholds, hide failures, or reinterpret failed controls as passing.
- The relative dedicated-control metric contract is recorded prospectively as raw balanced-accuracy ratio metadata on newly generated dedicated-control artifacts.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)
    return result

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
if unit_result.returncode != 0:
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

from pathlib import Path
import hashlib
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

EXPECTED_RELATIVE_METRIC_FORMULA_ID = 'raw_ba_ratio'
EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION = 'control_balanced_accuracy / baseline_balanced_accuracy'
REQUIRED_FORMULA_METADATA_CONTROLS = ['nuisance_shared_control', 'spatial_control']

# Pin reviewed source runs. Use None only if you intentionally want latest.txt resolution.
FEATURE_MATRIX_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase1_final_feature_matrix/20260421T151617731994Z')
COMPARATOR_RECONCILIATION_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase1_final_comparator_reconciliation/20260422T014337472987Z')

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_dedicated_controls'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_dedicated_controls_plan'

RUN_FINAL_DEDICATED_CONTROLS = False
REQUIRED_ACK = 'I reviewed final dedicated controls and understand failed controls must remain blockers without changing thresholds or opening claims'
MANUAL_LAUNCH_ACK = ''
MAX_OUTER_FOLDS = None  # Use all final LOSO folds by default.

FIX_MARKER = 'phase1_final_dedicated_controls_v2_formula_metadata_20260423'
print('Notebook fix marker:', FIX_MARKER)
print('Expected relative metric formula:', EXPECTED_RELATIVE_METRIC_FORMULA_ID)
print('Expected relative metric definition:', EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION)


In [ ]:
# Cell 3 - Resolve paths and validate prereg, feature-matrix, and comparator boundaries.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_latest(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        return Path(latest.read_text(encoding='utf-8').strip())
    runs = sorted([p for p in root.iterdir() if p.is_dir()]) if root.exists() else []
    if not runs:
        raise FileNotFoundError(f'No runs found under {root}')
    return runs[-1]

def resolve_prereg_bundle(path: Path) -> Path:
    path = Path(path)
    if path.exists():
        return path
    candidates = sorted(ARTIFACT_ROOT.glob('prereg/*/prereg_bundle.json'))
    print('Configured prereg bundle not found:', path)
    print('Found prereg bundles:', len(candidates))
    for item in candidates[-5:]:
        print('candidate:', item)
    if not candidates:
        raise FileNotFoundError(f'No prereg_bundle.json found under {ARTIFACT_ROOT / "prereg"}')
    return candidates[-1]

PREREG_BUNDLE = resolve_prereg_bundle(Path(PREREG_BUNDLE))
FEATURE_MATRIX_RUN = Path(FEATURE_MATRIX_RUN) if FEATURE_MATRIX_RUN else resolve_latest(ARTIFACT_ROOT / 'phase1_final_feature_matrix')
COMPARATOR_RECONCILIATION_RUN = Path(COMPARATOR_RECONCILIATION_RUN) if COMPARATOR_RECONCILIATION_RUN else resolve_latest(ARTIFACT_ROOT / 'phase1_final_comparator_reconciliation')

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

fm_summary = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_summary.json')
fm_validation = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_validation.json')
comp_summary = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciliation_summary.json')
comp_claim_state = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciled_claim_state.json')
assert fm_summary.get('status') == 'phase1_final_feature_matrix_materialized', fm_summary.get('status')
assert fm_summary.get('feature_matrix_ready') is True
assert fm_validation.get('status') == 'phase1_final_feature_matrix_validation_passed', fm_validation.get('status')
assert fm_summary.get('contains_logits') is False
assert fm_summary.get('contains_metrics') is False
assert comp_summary.get('status') == 'phase1_final_comparator_reconciliation_complete_claim_closed', comp_summary.get('status')
assert comp_summary.get('all_final_comparator_outputs_present') is True
assert comp_summary.get('runtime_comparator_logs_audited_for_all_required_comparators') is True
assert comp_summary.get('claim_ready') is False
assert comp_claim_state.get('claim_ready') is False
print('Feature matrix run:', FEATURE_MATRIX_RUN)
print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)


In [ ]:
# Cell 4 - Preflight runner/config availability and write a reviewable plan artifact.

from datetime import datetime, timezone

sys.path.insert(0, str(REPO_DIR))
preflight_blockers = []
try:
    from src.phase1.final_dedicated_controls import run_phase1_final_dedicated_controls  # noqa: F401
    runner_available = True
except Exception as exc:
    runner_available = False
    preflight_blockers.append(f'final dedicated controls runner import failed: {exc}')

required_repo_files = [
    REPO_DIR / 'src/phase1/final_dedicated_controls.py',
    REPO_DIR / 'configs/phase1/final_dedicated_controls.json',
    REPO_DIR / 'configs/phase1/final_comparator_runner.json',
    REPO_DIR / 'configs/gate2/synthetic_validation.json',
]
for path in required_repo_files:
    if not path.exists():
        preflight_blockers.append(f'missing repo file: {path.relative_to(REPO_DIR)}')

PLAN_ROOT.mkdir(parents=True, exist_ok=True)
plan_dir = PLAN_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir.mkdir(parents=True, exist_ok=False)

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_dedicated_controls',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--feature-matrix-run', str(FEATURE_MATRIX_RUN),
    '--comparator-reconciliation-run', str(COMPARATOR_RECONCILIATION_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--dedicated-controls-config', 'configs/phase1/final_dedicated_controls.json',
    '--comparator-runner-config', 'configs/phase1/final_comparator_runner.json',
    '--gate2-config', 'configs/gate2/synthetic_validation.json',
]
if MAX_OUTER_FOLDS is not None:
    cmd.extend(['--max-outer-folds', str(MAX_OUTER_FOLDS)])

plan = {
    'status': 'phase1_final_dedicated_controls_plan_recorded',
    'created_utc': plan_dir.name,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_DEDICATED_CONTROLS,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight_blockers,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'feature_matrix_run': str(FEATURE_MATRIX_RUN),
    'comparator_reconciliation_run': str(COMPARATOR_RECONCILIATION_RUN),
    'output_root': str(OUTPUT_ROOT),
    'expected_relative_metric_formula_id': EXPECTED_RELATIVE_METRIC_FORMULA_ID,
    'expected_relative_metric_formula_definition': EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION,
    'required_formula_metadata_controls': REQUIRED_FORMULA_METADATA_CONTROLS,
    'command': cmd,
    'scientific_integrity_rule': 'Plan only. Failed controls must stay failed; thresholds must not be edited in this notebook.',
}
(plan_dir / 'phase1_final_dedicated_controls_plan.json').write_text(json.dumps(plan, indent=2) + '\n', encoding='utf-8')
print(json.dumps({k: plan[k] for k in ['status', 'runner_available', 'run_flag', 'ack_matched', 'preflight_blockers', 'expected_relative_metric_formula_id']}, indent=2))
if preflight_blockers:
    raise RuntimeError(preflight_blockers)


In [ ]:
# Cell 5 - Manual hold or launch the CLI-backed dedicated controls runner.

if not RUN_FINAL_DEDICATED_CONTROLS:
    hold = {
        'status': 'phase1_final_dedicated_controls_manual_hold',
        'run_flag': RUN_FINAL_DEDICATED_CONTROLS,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'plan_dir': str(plan_dir),
    }
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch dedicated controls without explicit non-claim acknowledgement.')
else:
    launch_review = {
        'status': 'phase1_final_dedicated_controls_launch_review_passed',
        'run_flag': RUN_FINAL_DEDICATED_CONTROLS,
        'ack_matched': True,
        'claim_ready_before_run': False,
        'expected_relative_metric_formula_id': EXPECTED_RELATIVE_METRIC_FORMULA_ID,
        'expected_relative_metric_formula_definition': EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION,
    }
    (plan_dir / 'phase1_final_dedicated_controls_launch_review.json').write_text(json.dumps(launch_review, indent=2) + '\n', encoding='utf-8')
    run(cmd, cwd=REPO_DIR)
    print('Final dedicated controls command completed. Review generated artifacts before rerunning final controls.')


In [ ]:
# Cell 6 - Inspect latest final dedicated controls output if execution was launched.

summary = None
latest_run = None
formula_metadata_by_control = {}
if RUN_FINAL_DEDICATED_CONTROLS:
    latest_run = resolve_latest(OUTPUT_ROOT)
    print('Latest final dedicated controls output:', latest_run)
    required_files = [
        'phase1_final_dedicated_controls_inputs.json',
        'phase1_final_dedicated_controls_summary.json',
        'phase1_final_dedicated_controls_report.md',
        'phase1_final_dedicated_controls_source_links.json',
        'phase1_final_dedicated_controls_input_validation.json',
        'nuisance_shared_control.json',
        'spatial_control.json',
        'shuffled_teacher_control.json',
        'time_shifted_teacher_control.json',
        'phase1_final_dedicated_controls_runtime_leakage_audit.json',
        'final_dedicated_control_manifest.json',
        'phase1_final_dedicated_controls_claim_state.json',
    ]
    for name in required_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_dedicated_controls_summary.json')
    print(json.dumps({
        'status': summary.get('status'),
        'dedicated_control_suite_passed': summary.get('dedicated_control_suite_passed'),
        'computed_dedicated_control_results': summary.get('computed_dedicated_control_results'),
        'failed_dedicated_control_results': summary.get('failed_dedicated_control_results'),
        'claim_blockers': summary.get('claim_blockers'),
    }, indent=2))
    for control_id, filename in {
        'nuisance_shared_control': 'nuisance_shared_control.json',
        'spatial_control': 'spatial_control.json',
    }.items():
        payload = read_json(latest_run / filename)
        threshold = payload.get('threshold', {})
        formula_metadata_by_control[control_id] = {
            'relative_metric_formula_id': threshold.get('relative_metric_formula_id'),
            'relative_metric_formula_definition': threshold.get('relative_metric_formula_definition'),
            'relative_metric_formula_source': threshold.get('relative_metric_formula_source'),
            'relative_to_baseline': threshold.get('relative_to_baseline'),
        }
    print('Relative metric formula metadata:')
    print(json.dumps(formula_metadata_by_control, indent=2))
else:
    print('Manual hold only; no final dedicated controls output inspected.')


In [ ]:
# Cell 7 - Assertions and non-claim review note.

if RUN_FINAL_DEDICATED_CONTROLS:
    assert summary is not None
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert summary.get('status') in ['phase1_final_dedicated_controls_complete_claim_closed', 'phase1_final_dedicated_controls_blocked']
    manifest = read_json(latest_run / 'final_dedicated_control_manifest.json')
    leakage = read_json(latest_run / 'phase1_final_dedicated_controls_runtime_leakage_audit.json')
    assert manifest.get('claim_ready') is False
    assert manifest.get('smoke_artifacts_promoted') is False
    assert manifest.get('teacher_used_at_inference') is False
    assert manifest.get('test_time_privileged_or_teacher_outputs_allowed') is False
    assert manifest.get('relative_metric_contract', {}).get('formula_id') == EXPECTED_RELATIVE_METRIC_FORMULA_ID
    assert manifest.get('relative_metric_contract', {}).get('definition') == EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION
    assert leakage.get('outer_test_subject_used_for_any_fit') is False
    for control_id in REQUIRED_FORMULA_METADATA_CONTROLS:
        metadata = formula_metadata_by_control.get(control_id, {})
        assert metadata.get('relative_metric_formula_id') == EXPECTED_RELATIVE_METRIC_FORMULA_ID, control_id
        assert metadata.get('relative_metric_formula_definition') == EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION, control_id
        assert metadata.get('relative_metric_formula_source'), control_id
    if summary.get('dedicated_control_suite_passed') is False:
        assert summary.get('claim_blockers'), 'Blocked dedicated controls must name blockers.'
    review_note = {
        'status': 'phase1_final_dedicated_controls_review_recorded',
        'reviewed_run': str(latest_run),
        'scope': 'dedicated final negative controls from final feature matrix and locked LOSO splits',
        'dedicated_control_suite_passed': summary.get('dedicated_control_suite_passed'),
        'relative_metric_formula_contract': {
            'formula_id': EXPECTED_RELATIVE_METRIC_FORMULA_ID,
            'definition': EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION,
            'artifact_metadata_by_control': formula_metadata_by_control,
            'non_retroactive': True,
        },
        'metrics_interpretation': {
            'allowed': 'Negative-control pass/fail diagnostics only.',
            'not_allowed': 'Do not use dedicated controls as decoder efficacy, privileged-transfer efficacy, or full Phase 1 evidence.',
        },
        'next_allowed_steps': [
            'Rerun final controls with final_dedicated_control_manifest if review is acceptable.',
            'Then rerun final governance reconciliation with the updated final_control_manifest.',
            'Do not open headline claims while calibration or influence blockers remain.',
        ],
        'not_ok_to_claim': [
            'decoder efficacy',
            'A2d efficacy',
            'A3/A4 efficacy',
            'A4 superiority',
            'privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    }
    (latest_run / 'phase1_final_dedicated_controls_review_note.json').write_text(json.dumps(review_note, indent=2) + '\n', encoding='utf-8')
    print('Review note written:', latest_run / 'phase1_final_dedicated_controls_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('No assertions beyond plan/manual hold because RUN_FINAL_DEDICATED_CONTROLS is False.')


In [ ]:
# Cell 8 - Closeout.

print('================ Phase 1 Final Dedicated Controls Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_DEDICATED_CONTROLS)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Feature matrix run:', FEATURE_MATRIX_RUN)
print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)
print('Expected relative metric formula:', EXPECTED_RELATIVE_METRIC_FORMULA_ID)
print('Expected relative metric definition:', EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION)
print('Preflight blockers:', preflight_blockers)
if RUN_FINAL_DEDICATED_CONTROLS:
    print('Latest final dedicated controls output:', latest_run)
    print('Dedicated control suite passed:', summary.get('dedicated_control_suite_passed'))
    print('Computed dedicated controls:', summary.get('computed_dedicated_control_results'))
    print('Failed dedicated controls:', summary.get('failed_dedicated_control_results'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('Relative metric metadata by control:', json.dumps(formula_metadata_by_control, indent=2))
    print('CHECK REQUIRED: Review final_dedicated_control_manifest.json before rerunning final controls with --dedicated-control-manifest.')
else:
    print('HELD: Runner appears available, but dedicated controls were not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit non-claim acknowledgement if appropriate.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
